In [13]:
import torch
import itertools
import os
import pyro
import pyro.distributions as dist
from pyro.infer import Trace_ELBO
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from einops import repeat

from pyro_cases.base_vae import BaseVAEwRegister
from pyro_cases.run import vae_dict

In [2]:
# elbo_result_dir = Path("/data/scratch/pduan/new_gcvi_output/gcvi_06-30_non_amortized_vae_output/")
elbo_result_dir = Path("/data/scratch/pduan/new_gcvi_output/gcvi_07-11_non_amortized_vae_larger_init_range_output/")
favi_result_dir = Path("/data/scratch/pduan/new_gcvi_output/gcvi_06-30_set_transformer_favi_output/")

In [3]:
# note that the same seed can generate different results on cpu and gpu
device = torch.device("cuda:7")

In [4]:
elbo_files = set(os.listdir(elbo_result_dir))
favi_files = set(os.listdir(favi_result_dir))

In [5]:
print(f"ELBO files: {len(elbo_files)}")
print(f"FAVI files: {len(favi_files)}")

ELBO files: 115
FAVI files: 117


In [6]:
common_files = elbo_files & favi_files
print(f"common files: {len(common_files)}")
print(f"in FAVI, not in ELBO: {favi_files - elbo_files}")
print(f"in ELBO, not in FAVI: {elbo_files - favi_files}")

common files: 115
in FAVI, not in ELBO: {'pyro_t_arm_wells_predicted_mn_96.pt', 'pyro_t_arm_wells_probit_mn_96.pt'}
in ELBO, not in FAVI: set()


In [7]:
valid_files = []
for cf in common_files:
    favi_results = torch.load(favi_result_dir / cf, map_location="cpu")
    elbo_results = torch.load(elbo_result_dir / cf, map_location="cpu")
    find_elbo_error = any([r["elbo_error"] is not None for r in elbo_results])
    find_favi_error = any([r["favi_error"] is not None for r in favi_results])
    if find_elbo_error or find_favi_error:
        continue
    valid_files.append(cf)

In [8]:
len(valid_files)

105

In [9]:
def get_est_mu_sigma2(results, tag):
    est_mu = []
    for r in results:
        est_mu.append(r[f"{tag}_test_dict_list"]["est_mu"])  # (num_obs, k)
    est_mu = torch.stack(est_mu, dim=0)  # (r, num_obs, k)
    est_sigma2 = []
    for r in results:
        est_sigma2.append(r[f"{tag}_test_dict_list"]["est_sigma2"])  # (num_obs, k)
    est_sigma2 = torch.stack(est_sigma2, dim=0)  # (r, num_obs, k)
    return est_mu, est_sigma2

In [11]:
def print_value(favi_value, elbo_value, tag):
    print(f"FAVI mean({tag}): {favi_value.mean():.3e}")
    print(f"ELBO mean({tag}): {elbo_value.mean():.3e}")
    print(f"FAVI median({tag}): {favi_value.median():.3e}")
    print(f"ELBO median({tag}): {elbo_value.median():.3e}")

def print_cases(mean_favi_g_elbo_cases, mean_favi_l_elbo_cases, median_favi_g_elbo_cases, median_favi_l_elbo_cases, tag):
    mean_g_c = len(mean_favi_g_elbo_cases)
    mean_l_c = len(mean_favi_l_elbo_cases)
    mean_deno = mean_g_c + mean_l_c
    median_g_c = len(median_favi_g_elbo_cases)
    median_l_c = len(median_favi_l_elbo_cases)
    median_deno = median_g_c + median_l_c
    print(f"favi mean({tag}) > elbo mean({tag}): {mean_g_c}/{mean_deno} ({mean_g_c / mean_deno:.3f})")
    print(f"favi mean({tag}) <= elbo mean({tag}): {mean_l_c}/{mean_deno} ({mean_l_c / mean_deno:.3f})")
    print(f"favi median({tag}) > elbo median({tag}): {median_g_c}/{median_deno} ({median_g_c / median_deno:.3f})")
    print(f"favi median({tag}) <= elbo median({tag}): {median_l_c}/{median_deno} ({median_l_c / median_deno:.3f})")

In [16]:
def energy_fn(est_dist, true_value, m=8):
    est_samples = est_dist.sample((m,))  # (m, r, b, k)
    first_term = (est_samples - true_value).abs().mean(dim=0)
    second_term = (est_samples.unsqueeze(1) - est_samples.unsqueeze(0)).abs().mean(dim=(0, 1)) * m / (m - 1)
    return first_term - 0.5 * second_term  # (r, b, k)

In [ ]:
mean_logp_favi_g_elbo_cases = []
median_logp_favi_g_elbo_cases = []
mean_logp_favi_l_elbo_cases = []
median_logp_favi_l_elbo_cases = []

mean_elbo_favi_g_elbo_cases = []
mean_elbo_favi_l_elbo_cases = []
median_elbo_favi_g_elbo_cases = []
median_elbo_favi_l_elbo_cases = []

mean_energy_favi_g_elbo_cases = []
mean_energy_favi_l_elbo_cases = []
median_energy_favi_g_elbo_cases = []
median_energy_favi_l_elbo_cases = []

cant_reproduce_cases = []

for i, vf in enumerate(valid_files):
    favi_results = torch.load(favi_result_dir / vf, map_location="cpu")
    elbo_results = torch.load(elbo_result_dir / vf, map_location="cpu")

    task_name = favi_results[0]["task"]
    assert task_name == elbo_results[0]["task"]
    favi_vae = vae_dict[task_name](hidden_dim=1024, use_neural_network=True, nn_type="set_transformer").to(device=device)
    elbo_vae = vae_dict[task_name](hidden_dim=1, use_neural_network=False).to(device=device)
    test_seed = favi_results[0]["favi_test_dict_list"]["obs_seed"]
    assert test_seed == elbo_results[0]["elbo_test_dict_list"]["obs_seed"]
    n_test_obs = favi_results[0]["favi_test_dict_list"]["est_mu"].shape[0]
    assert n_test_obs == elbo_results[0]["elbo_test_dict_list"]["est_mu"].shape[0]
    if isinstance(favi_vae, BaseVAEwRegister):
        favi_vae.do_register(n_test_obs)
    if isinstance(elbo_vae, BaseVAEwRegister):
        elbo_vae.do_register(n_test_obs)
    pyro.set_rng_seed(test_seed)
    test_sample_dict = favi_vae.generate_sample_dict(batch_size=n_test_obs)

    print("=" * 50)
    print(f"[{i + 1}] task name: {task_name}")

    # test reproducibility
    unequal_flag = False
    favi_obs, favi_true_theta = favi_vae.extract_x_for_set_transformer(n_test_obs, test_sample_dict), favi_vae.extract_theta(test_sample_dict)
    favi_obs = favi_obs.cpu()
    favi_true_theta = favi_true_theta.cpu()
    if not torch.allclose(favi_results[0]["favi_test_dict_list"]["obs"], favi_obs):
        print("unequal favi obs")
        unequal_flag = True
    if not torch.allclose(favi_results[0]["favi_test_dict_list"]["true_theta"], favi_true_theta):
        print("unequal favi true_theta")
        unequal_flag = True
    elbo_obs, elbo_true_theta = elbo_vae.extract_x(test_sample_dict), favi_vae.extract_theta(test_sample_dict)
    elbo_obs = elbo_obs.cpu()
    elbo_true_theta = elbo_true_theta.cpu()
    if not torch.allclose(elbo_results[0]["elbo_test_dict_list"]["obs"], elbo_obs):
        print("unequal elbo obs")
        unequal_flag = True
    if not torch.allclose(elbo_results[0]["elbo_test_dict_list"]["true_theta"], elbo_true_theta):
        print("unequal elbo true_theta")
        unequal_flag = True
    if unequal_flag:
        cant_reproduce_cases.append(task_name)
        continue
    
    # test logp
    favi_est_mu, favi_est_sigma2 = get_est_mu_sigma2(favi_results, tag="favi")
    elbo_est_mu, elbo_est_sigma2 = get_est_mu_sigma2(elbo_results, tag="elbo")
    favi_logp = dist.Normal(favi_est_mu, favi_est_sigma2.sqrt()).log_prob(repeat(favi_true_theta, "b k -> r b k", r=len(favi_results)))
    elbo_logp = dist.Normal(elbo_est_mu, elbo_est_sigma2.sqrt()).log_prob(repeat(elbo_true_theta, "b k -> r b k", r=len(elbo_results)))
    print_value(favi_logp, elbo_logp, tag="logp")
    if favi_logp.mean() > elbo_logp.mean():
        mean_logp_favi_g_elbo_cases.append(task_name)
    else:
        mean_logp_favi_l_elbo_cases.append(task_name)
    if favi_logp.median() > elbo_logp.median():
        median_logp_favi_g_elbo_cases.append(task_name)
    else:
        median_logp_favi_l_elbo_cases.append(task_name)

    # test elbo
    favi_elbo_value = []
    for sub_favi_est_mu, sub_favi_est_sigma2 in zip(favi_est_mu, favi_est_sigma2, strict=True):
        favi_vae.set_theta_loc_scale(theta_loc=sub_favi_est_mu.to(device=device), 
                                     theta_scale=sub_favi_est_sigma2.sqrt().to(device=device))
        favi_elbo_value.append(-1 * Trace_ELBO(num_particles=1).loss(favi_vae.model, 
                                                                     favi_vae.guide, 
                                                                     n_test_obs, 
                                                                     test_sample_dict))
    favi_elbo_value = torch.tensor(favi_elbo_value)
    elbo_elbo_value = []
    for sub_elbo_est_mu, sub_elbo_est_sigma2 in zip(elbo_est_mu, elbo_est_sigma2, strict=True):
        elbo_vae.set_theta_loc_scale(theta_loc=sub_elbo_est_mu.to(device=device), 
                                     theta_scale=sub_elbo_est_sigma2.sqrt().to(device=device))
        elbo_elbo_value.append(-1 * Trace_ELBO(num_particles=1).loss(elbo_vae.model, 
                                                                     elbo_vae.guide, 
                                                                     n_test_obs, 
                                                                     test_sample_dict))
    elbo_elbo_value = torch.tensor(elbo_elbo_value)
    print_value(favi_elbo_value, elbo_elbo_value, tag="elbo")
    if favi_elbo_value.mean() > elbo_elbo_value.mean():
        mean_elbo_favi_g_elbo_cases.append(task_name)
    else:
        mean_elbo_favi_l_elbo_cases.append(task_name)
    if favi_elbo_value.median() > elbo_elbo_value.median():
        median_elbo_favi_g_elbo_cases.append(task_name)
    else:
        median_elbo_favi_l_elbo_cases.append(task_name)

    # test energy
    favi_energy = energy_fn(dist.Normal(favi_est_mu, favi_est_sigma2.sqrt()), favi_true_theta)
    elbo_energy = energy_fn(dist.Normal(elbo_est_mu, elbo_est_sigma2.sqrt()), elbo_true_theta)
    print_value(favi_energy, elbo_energy, tag="energy")
    if favi_energy.mean() > elbo_energy.mean():
        mean_energy_favi_g_elbo_cases.append(task_name)
    else:
        mean_energy_favi_l_elbo_cases.append(task_name)
    if favi_energy.median() > elbo_energy.median():
        median_energy_favi_g_elbo_cases.append(task_name)
    else:
        median_energy_favi_l_elbo_cases.append(task_name)

print()
print("+" * 100)
print("Summary:")
print(f"can't reproduce cases: {len(cant_reproduce_cases)} ({cant_reproduce_cases})")
print_cases(mean_logp_favi_g_elbo_cases, 
            mean_logp_favi_l_elbo_cases, 
            median_logp_favi_g_elbo_cases, 
            median_logp_favi_l_elbo_cases, 
            tag="logp")
print_cases(mean_elbo_favi_g_elbo_cases, 
            mean_elbo_favi_l_elbo_cases, 
            median_elbo_favi_g_elbo_cases, 
            median_elbo_favi_l_elbo_cases, 
            tag="elbo")
print_cases(mean_energy_favi_g_elbo_cases, 
            mean_energy_favi_l_elbo_cases, 
            median_energy_favi_g_elbo_cases, 
            median_energy_favi_l_elbo_cases, 
            tag="energy")
print("+" * 100)

[1] task name: arm_logearn_interaction
FAVI mean(logp): -6.764e+00
ELBO mean(logp): -9.916e+00
FAVI median(logp): -1.770e+00
ELBO median(logp): -4.304e+00
FAVI mean(elbo): -4.981e+11
ELBO mean(elbo): -2.399e+13
FAVI median(elbo): -5.063e+11
ELBO median(elbo): -2.389e+13
FAVI mean(energy): 3.764e+00
ELBO mean(energy): 9.577e+00
FAVI median(energy): 8.204e-01
ELBO median(energy): 7.471e+00
[2] task name: arm_wells_dae_inter
FAVI mean(logp): 4.511e-01
ELBO mean(logp): -3.288e+00
FAVI median(logp): 6.156e-01
ELBO median(logp): -2.932e+00
FAVI mean(elbo): -3.005e+04
ELBO mean(elbo): -4.433e+06
FAVI median(elbo): -3.005e+04
ELBO median(elbo): -4.426e+06
FAVI mean(energy): 9.122e-02
ELBO mean(energy): 2.589e+00
FAVI median(energy): 6.265e-02
ELBO median(energy): 2.099e+00
[3] task name: arm_electric_1b
FAVI mean(logp): -6.745e+02
ELBO mean(logp): -4.871e+02
FAVI median(logp): -1.237e+02
ELBO median(logp): -5.901e+00
FAVI mean(elbo): -3.928e+11
ELBO mean(elbo): -2.606e+13
FAVI median(elbo): -3